# 02 Materialize training snapshot from Athena

Goal: materialize one deterministic ECMWF t2m snapshot from Athena into `data/t2m_snapshot.csv`, inspect it, and keep the logic aligned with `src/query_athena.py`.

This notebook intentionally mirrors the current repository baseline:
- source: `silverlayer.forecast_parquet`
- filters: `variable = 't'`, `isobaricInhPa = 1000.0`
- partition filters: `year`, `month`, `day`, `hour`
- output schema: `latitude`, `longitude`, `value`


## Usage notes

Before running:
1. Ensure `.env` exists at the repo root.
2. Ensure AWS credentials are available in the current environment.
3. Adjust the partition values in the parameters cell if needed.

Expected `.env` variables:
- `REGION`
- `ATHENA_DATABASE`
- `ATHENA_TABLE`
- `ATHENA_OUTPUT_BUCKET`
- `ATHENA_OUTPUT_PREFIX`
- optional: `AWS_PROFILE`


In [ ]:
from pathlib import Path
import io
import os
import time
from urllib.parse import urlparse

import boto3
import pandas as pd
import matplotlib.pyplot as plt
from botocore.exceptions import BotoCoreError, ClientError
from dotenv import load_dotenv

plt.rcParams['figure.figsize'] = (8, 6)
pd.options.display.max_rows = 20
pd.options.display.max_columns = 20


In [ ]:
# Repo root resolution
repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    if (repo_root.parent / 'src').exists():
        repo_root = repo_root.parent
    else:
        raise RuntimeError('Could not locate repo root. Run this notebook from the repository or notebooks/ directory.')

print(f'Repo root: {repo_root}')
load_dotenv(repo_root / '.env')


In [ ]:
# Parameters: keep these aligned with src/query_athena.py defaults unless you intentionally change them.
YEAR = '2026'
MONTH = '04'
DAY = '09'
HOUR = '18'

OUTPUT_CSV = repo_root / 'data' / 't2m_snapshot.csv'
SHOW_PLOT = True


In [ ]:
def required_env(name: str) -> str:
    value = os.getenv(name, '').strip()
    if not value:
        raise ValueError(f'Missing required environment variable: {name}')
    return value


def parse_s3_uri(s3_uri: str):
    parsed = urlparse(s3_uri)
    if parsed.scheme != 's3' or not parsed.netloc or not parsed.path:
        raise ValueError(f'Invalid S3 URI: {s3_uri}')
    return parsed.netloc, parsed.path.lstrip('/')


def build_query(database: str, table: str, year: str, month: str, day: str, hour: str) -> str:
    # Intentionally mirrors src/query_athena.py
    return f'''
SELECT latitude, longitude, value
FROM {database}.{table}
WHERE variable = 't'
  AND isobaricInhPa = 1000.0
  AND year = '{year}'
  AND month = '{month}'
  AND day = '{day}'
  AND hour = '{hour}'
'''.strip()


In [ ]:
REGION = required_env('REGION')
DATABASE = required_env('ATHENA_DATABASE')
TABLE = required_env('ATHENA_TABLE')
ATHENA_OUTPUT_BUCKET = required_env('ATHENA_OUTPUT_BUCKET')
ATHENA_OUTPUT_PREFIX = required_env('ATHENA_OUTPUT_PREFIX').strip('/')
AWS_PROFILE = os.getenv('AWS_PROFILE', '').strip()

print({
    'REGION': REGION,
    'ATHENA_DATABASE': DATABASE,
    'ATHENA_TABLE': TABLE,
    'ATHENA_OUTPUT_BUCKET': ATHENA_OUTPUT_BUCKET,
    'ATHENA_OUTPUT_PREFIX': ATHENA_OUTPUT_PREFIX,
    'AWS_PROFILE': AWS_PROFILE or None,
    'PARTITIONS': {'year': YEAR, 'month': MONTH, 'day': DAY, 'hour': HOUR},
    'OUTPUT_CSV': str(OUTPUT_CSV),
})


In [ ]:
query = build_query(DATABASE, TABLE, YEAR, MONTH, DAY, HOUR)
print(query)


In [ ]:
session_kwargs = {'region_name': REGION}
if AWS_PROFILE:
    session_kwargs['profile_name'] = AWS_PROFILE

session = boto3.Session(**session_kwargs)
athena = session.client('athena')
s3 = session.client('s3')
output_location = f's3://{ATHENA_OUTPUT_BUCKET}/{ATHENA_OUTPUT_PREFIX}/'

print(f'Athena output location: {output_location}')


In [ ]:
start = athena.start_query_execution(
    QueryString=query,
    QueryExecutionContext={'Database': DATABASE},
    ResultConfiguration={'OutputLocation': output_location},
)
query_id = start['QueryExecutionId']
print(f'QueryExecutionId: {query_id}')


In [ ]:
final_state = None
failure_reason = None
execution = None

for _ in range(300):
    execution = athena.get_query_execution(QueryExecutionId=query_id)
    status = execution['QueryExecution']['Status']
    final_state = status['State']
    if final_state in {'SUCCEEDED', 'FAILED', 'CANCELLED'}:
        failure_reason = status.get('StateChangeReason')
        break
    time.sleep(2)

print({'state': final_state, 'reason': failure_reason})
if final_state != 'SUCCEEDED':
    raise RuntimeError(
        f"Athena query failed with state={final_state}. Reason: {failure_reason or 'not provided'}"
    )


In [ ]:
result_uri = execution['QueryExecution']['ResultConfiguration']['OutputLocation']
result_bucket, result_key = parse_s3_uri(result_uri)
print({'result_uri': result_uri, 'bucket': result_bucket, 'key': result_key})

response = s3.get_object(Bucket=result_bucket, Key=result_key)
body = response['Body'].read()
df = pd.read_csv(io.BytesIO(body))

print(f'Raw rows returned by Athena: {len(df)}')
df.head()


In [ ]:
EXPECTED_COLUMNS = {'latitude', 'longitude', 'value'}
missing_columns = sorted(EXPECTED_COLUMNS - set(df.columns))
if missing_columns:
    raise ValueError(
        'Athena result missing required columns: '
        + ', '.join(missing_columns)
        + '. Expected: latitude, longitude, value'
    )

df = df[['latitude', 'longitude', 'value']].dropna()
if df.empty:
    raise ValueError('Athena query returned zero usable rows after filtering/selecting latitude, longitude, value')

print(f'Usable rows after validation: {len(df)}')
df.describe()


In [ ]:
null_counts = df.isna().sum().to_dict()
print({'null_counts': null_counts})
df.head(10)


In [ ]:
if SHOW_PLOT:
    ax = df.plot.scatter(x='longitude', y='latitude', c='value', colormap='viridis', s=8)
    ax.set_title(f'T2M snapshot {YEAR}-{MONTH}-{DAY} {HOUR}:00')
    plt.show()


In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f'Snapshot written to: {OUTPUT_CSV}')


## Next step

Once this notebook succeeds, the next repo-native step is:

```bash
python src/prepare_training_data.py --input data/t2m_snapshot.csv --output data/training/train.csv --dataset-name 2026-04-09-18
```
